# BEE 4750 Homework 5: Mixed Integer and Stochastic Programming

**Name**:Jerry Lu

**ID**:jl3498

> **Due Date**
>
> Thursday, 12/04/24, 9:00pm

## Overview

### Instructions

-   In Problem 1, you will use mixed integer programming to solve a
    waste load allocation problem.
-   In Problem 2, you will formulate a stochastic optimization problem.

### Load Environment

The following code loads the environment and makes sure all needed
packages are installed. This should be at the start of most Julia
scripts.

In [2]:
import Pkg
Pkg.activate(@__DIR__)
Pkg.instantiate()

  Activating project at `c:\Users\shini\hw5-jl3498-hw5`


In [3]:
using JuMP
using HiGHS
using DataFrames
using GraphRecipes
using Plots
using Measures
using MarkdownTables

## Problems (Total: 30 Points)

### Problem 1 (24 points)

Three cities are developing a coordinated municipal solid waste (MSW)
disposal plan. Three disposal alternatives are being considered: a
landfill (LF), a materials recycling facility (MRF), and a
waste-to-energy facility (WTE). The capacities of these facilities and
the fees for operation and disposal are provided below.

-   **LF**: Capacity 200 Mg, fixed cost \$2000/day, tipping cost
    \$50/Mg;
-   **MRF**: Capacity 350 Mg, fixed cost \$1500/day, tipping cost
    \$7/Mg, recycling cost \$40/Mg recycled;
-   **WTE**: Capacity 210 Mg, fixed cost \$2500/day, tipping cost
    \$60/Mg;

The MRF recycling rate is 40%, and the ash fraction of non-recycled
waste is 16% and of recycled waste is 14%. Transportation costs are
\$1.5/Mg-km, and the relative distances between the cities and
facilities are provided in the table below.

| **City/Facility** | **Landfill (km)** | **MRF (km)** | **WTE (km)** |
|:-----------------:|:-----------------:|:------------:|:------------:|
|         1         |         5         |      30      |      15      |
|         2         |        15         |      25      |      10      |
|         3         |        13         |      45      |      20      |
|        LF         |        \-         |      32      |      18      |
|        MRF        |        32         |      \-      |      15      |
|        WTE        |        18         |      15      |      \-      |

The fixed costs associated with the disposal options are incurred only
if the particular disposal option is implemented. The three cities
produce 100, 90, and 120 Mg/day of solid waste, respectively, with the
composition provided in the table below.

| **Component** | **% of total mass** | **Combustion ash** (%) | **MRF Recycling rate** (%) |
|:---------------------:|:--------------:|:---------------:|:---------------:|
| Food Wastes | 15 | 8 | 0 |
| Paper & Cardboard | 40 | 7 | 55 |
| Plastics | 5 | 5 | 15 |
| Textiles | 3 | 10 | 10 |
| Rubber, Leather | 2 | 15 | 0 |
| Wood | 5 | 2 | 30 |
| Yard Wastes | 18 | 2 | 40 |
| Glass | 4 | 100 | 60 |
| Ferrous | 2 | 100 | 75 |
| Aluminum | 2 | 100 | 80 |
| Other Metal | 1 | 100 | 50 |
| Miscellaneous | 3 | 70 | 0 |

The information in the above table will help you determine the overall
recycling and ash fractions. Note that the recycling residuals, which
may be sent to either landfill or the WTE, have different ash content
than the ash content of the original MSW. You will need to determine
these fractions to construct your mass balance constraints.

**Reminder**: Use `round(x; digits=n)` to report values to the
appropriate precision!

#### Problem 1.1

Based on the information above, calculate the overall recycling and ash
fractions for the waste produced by each city.

In [4]:
# Waste composition data
components = [
    ("Food Wastes",        15,  8,   0),
    ("Paper & Cardboard",  40,  7,  55),
    ("Plastics",            5,  5,  15),
    ("Textiles",            3, 10,  10),
    ("Rubber, Leather",     2, 15,   0),
    ("Wood",                5,  2,  30),
    ("Yard Wastes",        18,  2,  40),
    ("Glass",               4, 100, 60),
    ("Ferrous",             2, 100, 75),
    ("Aluminum",            2, 100, 80),
    ("Other Metal",         1, 100, 50),
    ("Miscellaneous",       3, 70,   0)
]

# compute totals
total_recycled = 0.0
total_ash = 0.0

for (name, mass_pct, ash_pct, rec_rate) in components
    frac = mass_pct/100
    recycle_frac = rec_rate/100
    ash_frac = ash_pct/100

    # amount recycled
    total_recycled += frac * recycle_frac

    # ash contribution (ash of total incoming waste)
    total_ash += frac * ash_frac
end

println("Overall recycling fraction = ", round(total_recycled; digits=4))
println("Overall ash fraction       = ", round(total_ash; digits=4))


Overall recycling fraction = 0.3775
Overall ash fraction       = 0.1641


This finds the overall recycling and ash fractions for each of the wastes produced through a for loop.

#### Problem 1.2

What are the decision variables for your optimization problem? Provide
notation and variable meaning.

## Decision Variables

- **Facility opening decisions**
  - $y_f \in \{0,1\}$  
    Indicates whether facility $f$ is opened.  
    - $y_f = 1$ if facility $f$ (LF, MRF, or WTE) is opened and its fixed cost is incurred  
    - $y_f = 0$ otherwise  

- **City-to-facility waste flows**
  - $x_{i,f} \ge 0$  
    Amount of waste shipped from city $i$ to facility $f$, where  
    $i \in \{1,2,3\}$ and $f \in \{\text{LF}, \text{MRF}, \text{WTE}\}$.

- **MRF residual routing**
  - $r_{\text{MRF}\to \text{LF}} \ge 0$  
    Amount of MRF residuals sent to the landfill.
  - $r_{\text{MRF}\to \text{WTE}} \ge 0$  
    Amount of MRF residuals sent to the WTE facility.

- **WTE ash routing**
  - $a_{\text{WTE}\to \text{LF}} \ge 0$  
    Amount of ash produced at the WTE facility that is sent to the landfill.

- **Facility throughputs** (auxiliary variables)
  - $m_{\text{MRF}} \ge 0$ — total mass entering the MRF  
  - $m_{\text{WTE}} \ge 0$ — total mass entering the WTE  
  - $m_{\text{LF}} \ge 0$ — total mass entering the landfill  
  - $m_{\text{rec}} \ge 0$ — mass recycled at the MRF  
  - $m_{\text{res}} \ge 0$ — mass of residuals produced by the MRF  

These variables determine which facilities are opened, how waste flows through the network, how much material is recycled or landfilled, and the resulting system cost.


#### Problem 1.3

Formulate the objective function. Make sure to include any needed
derivations or justifications for your equation(s).

## Objective Function — Total Daily System Cost

### Parameters 
- Facilities: $f \in \{\text{LF}, \text{MRF}, \text{WTE}\}$  
- Fixed daily cost of opening facility $f$: $F_f$ (\$/day)  
  - $F_{LF}=2000$, $F_{MRF}=1500$, $F_{WTE}=2500$  
- Tipping (disposal) cost at facility $f$: $c_f$ (\$/Mg)  
  - $c_{LF}=50$, $c_{MRF}=7$, $c_{WTE}=60$  
- Recycling processing cost at MRF per Mg recycled: $c_{\text{rec}}$ (\$/Mg)  
  - $c_{\text{rec}}=40$  
- Transportation cost per Mg·km: $\tau$ (\$/Mg·km)  
  - $\tau = 1.5$  

---

### Derivation — Cost Components

1. **Fixed facility costs:**  
   Only incurred if the facility is opened.  
   $$
   \sum_{f \in \{\text{LF}, \text{MRF}, \text{WTE}\}} F_f \, y_f
   $$

2. **Tipping / disposal costs:**  
   - Raw MSW shipped from cities to facilities:  
     $$
     \sum_{i=1}^{3} \sum_{f \in \{\text{LF}, \text{MRF}, \text{WTE}\}} c_f \, x_{i,f}
     $$  
   - MRF residuals sent to landfill or WTE:  
     $$
     c_{\text{LF}} \, r_{\text{MRF}\to \text{LF}} + c_{\text{WTE}} \, r_{\text{MRF}\to \text{WTE}}
     $$  
   - WTE ash sent to landfill:  
     $$
     c_{\text{LF}} \, a_{\text{WTE}\to \text{LF}}
     $$

3. **MRF recycling processing cost:**  
   $$
   c_{\text{rec}} \, m_{\text{rec}} = c_{\text{rec}} \, R_{\text{MRF}} \, m_{\text{MRF}}
   $$

4. **Transportation costs:**  
   - Cities → facilities:  
     $$
     \tau \sum_{i=1}^{3} \sum_{f \in \{\text{LF}, \text{MRF}, \text{WTE}\}} x_{i,f} \, d_{i,f}
     $$  
   - Inter-facility flows (MRF residuals, WTE ash):  
     $$
     \tau \left( r_{\text{MRF}\to \text{LF}} \, d_{\text{MRF,LF}} + r_{\text{MRF}\to \text{WTE}} \, d_{\text{MRF,WTE}} + a_{\text{WTE}\to \text{LF}} \, d_{\text{WTE,LF}} \right)
     $$

---

### Full Objective

Minimize total daily system cost $Z$:

$$
\begin{aligned}
\min Z = \; &
\underbrace{\sum_{f} F_f \, y_f}_{\text{fixed facility costs}} \;+\;
\underbrace{\sum_{i=1}^{3} \sum_{f} c_f \, x_{i,f}}_{\text{tipping of raw MSW}} \;+\;
\underbrace{c_{\text{rec}} \, R_{\text{MRF}} \, m_{\text{MRF}}}_{\text{MRF recycling cost}} \\
&+\underbrace{c_{\text{LF}} \, r_{\text{MRF}\to \text{LF}} + c_{\text{WTE}} \, r_{\text{MRF}\to \text{WTE}}}_{\text{tipping of MRF residuals}} \;+\;
\underbrace{c_{\text{LF}} \, a_{\text{WTE}\to \text{LF}}}_{\text{tipping of WTE ash}} \\
&+\underbrace{\tau \sum_{i=1}^{3} \sum_{f} x_{i,f} \, d_{i,f}}_{\text{transport: cities → facilities}} \;+\;
\underbrace{\tau \left( r_{\text{MRF}\to \text{LF}} \, d_{\text{MRF,LF}} + r_{\text{MRF}\to \text{WTE}} \, d_{\text{MRF,WTE}} + a_{\text{WTE}\to \text{LF}} \, d_{\text{WTE,LF}} \right)}_{\text{transport: inter-facility / ash}}
\end{aligned}
$$

---

### Auxiliary Relationships

- Total mass entering MRF: $m_{\text{MRF}} = \sum_{i=1}^{3} x_{i,\text{MRF}}$  
- Total mass recycled at MRF: $m_{\text{rec}} = R_{\text{MRF}} \, m_{\text{MRF}}$  
- Residual mass leaving MRF: $m_{\text{res}} = (1 - R_{\text{MRF}}) \, m_{\text{MRF}}$  
- Residual allocation: $r_{\text{MRF}\to \text{LF}} + r_{\text{MRF}\to \text{WTE}} = m_{\text{res}}$  
- Ash from WTE: $a_{\text{WTE}\to \text{LF}} = \alpha_{\text{raw}} \sum_{i=1}^{3} x_{i,\text{WTE}} + \alpha_{\text{res}} \, r_{\text{MRF}\to \text{WTE}}$


#### Problem 1.4

Derive all relevant constraints. Make sure to include any needed
justifications or derivations.

## Constraints — Mass Balances, Capacities, and Linking

### 1) City supply balances

All waste produced by each city must be assigned to a facility:

$$
\sum_{f \in \{\text{LF}, \text{MRF}, \text{WTE}\}} x_{i,f} = S_i, \quad i = 1,2,3
$$

Ensures that all waste generated by a city is sent to some facility (mass conservation).

---

### 2) Facility throughput definitions

Total mass entering each facility:

- **MRF:**  
$$
m_{\text{MRF}} = \sum_{i=1}^{3} x_{i,\text{MRF}}
$$

- **WTE:**  
$$
m_{\text{WTE}} = \sum_{i=1}^{3} x_{i,\text{WTE}} + r_{\text{MRF}\to \text{WTE}}
$$

- **Landfill:**  
$$
m_{\text{LF}} = \sum_{i=1}^{3} x_{i,\text{LF}} + r_{\text{MRF}\to \text{LF}} + a_{\text{WTE}\to \text{LF}}
$$

Defines the total mass processed by each facility, which will be used for capacity constraints and cost calculation.

---

### 3) MRF recycling and residuals

- **Recycled mass:**  
$$
m_{\text{rec}} = R_{\text{MRF}} \, m_{\text{MRF}}
$$

- **Residual (non-recycled) mass:**  
$$
m_{\text{res}} = (1 - R_{\text{MRF}}) \, m_{\text{MRF}}
$$

- **Residual allocation:**  
$$
r_{\text{MRF}\to \text{LF}} + r_{\text{MRF}\to \text{WTE}} = m_{\text{res}}
$$

with
$$
r_{\text{MRF}\to \text{LF}} \ge 0, \quad r_{\text{MRF}\to \text{WTE}} \ge 0
$$

The MRF splits incoming mass into recycled material and residuals; residuals must be assigned to a downstream facility.

---

### 4) WTE ash generation and routing

All ash produced at the WTE is sent to the landfill:

$$
a_{\text{WTE}\to \text{LF}} = \alpha_{\text{raw}} \sum_{i=1}^{3} x_{i,\text{WTE}} + \alpha_{\text{res}} \, r_{\text{MRF}\to \text{WTE}}
$$

Ash fractions differ for raw MSW and residuals; the formula calculates total ash directed to the landfill.

---

### 5) Facility capacity constraints

Each facility cannot exceed its processing capacity, and flows are only allowed if the facility is opened:

- **Landfill:**  
$$
\sum_{i=1}^{3} x_{i,\text{LF}} + r_{\text{MRF}\to \text{LF}} + a_{\text{WTE}\to \text{LF}} \le C_{\text{LF}} \, y_{\text{LF}}
$$

- **MRF:**  
$$
\sum_{i=1}^{3} x_{i,\text{MRF}} \le C_{\text{MRF}} \, y_{\text{MRF}}
$$

- **WTE:**  
$$
\sum_{i=1}^{3} x_{i,\text{WTE}} + r_{\text{MRF}\to \text{WTE}} \le C_{\text{WTE}} \, y_{\text{WTE}}
$$

Standard capacity constraints. If $y_f = 0$, the RHS is zero, preventing flow to a closed facility.

---

### 6) Linking constraints for flows

To prevent shipments to closed facilities:

$$
\sum_{i=1}^{3} x_{i,f} \le C_f \, y_f, \quad f \in \{\text{LF}, \text{MRF}, \text{WTE}\}
$$

Residuals and ash cannot be routed to closed facilities:

$$
r_{\text{MRF}\to \text{LF}} \le C_{\text{LF}} \, y_{\text{LF}}, \quad
r_{\text{MRF}\to \text{WTE}} \le C_{\text{WTE}} \, y_{\text{WTE}}, \quad
a_{\text{WTE}\to \text{LF}} \le C_{\text{LF}} \, y_{\text{LF}}
$$

Big-M constraints ensure flows only occur if the facility is open; using $C_f$ as $M_f$ keeps the bound tight.

---

### 7) Nonnegativity

$$
x_{i,f} \ge 0, \quad r_{\text{MRF}\to \text{LF}} \ge 0, \quad r_{\text{MRF}\to \text{WTE}} \ge 0, \quad a_{\text{WTE}\to \text{LF}} \ge 0
$$

$$
y_f \in \{0,1\}, \quad f \in \{\text{LF}, \text{MRF}, \text{WTE}\}
$$

Waste flows cannot be negative; facility opening decisions are binary.

---

### 8) policy constraints

- Minimum recycling target:  
$$
m_{\text{rec}} = R_{\text{MRF}} \, m_{\text{MRF}} \ge T
$$

- Force facility to open:  
$$
y_{\text{LF}} = 1
$$

- Mutually exclusive facility selection:  
$$
\sum_{f \in \text{MRF candidates}} y_f \le 1
$$


#### Problem 1.5

Find the optimal solution (using `JuMP` to solve the problem). Report
the optimal objective value.

In [ ]:

# Given data
cities = 1:3
facilities = [:LF, :MRF, :WTE]
# City waste (Mg/day)
S = Dict(1=>100, 2=>90, 3=>120)
# Facility capacities (Mg/day)
C = Dict(:LF=>200, :MRF=>350, :WTE=>210)
# Fixed costs ($/day)
F = Dict(:LF=>2000, :MRF=>1500, :WTE=>2500)
# Tipping costs ($/Mg)
c = Dict(:LF=>50, :MRF=>7, :WTE=>60)
# Recycling processing cost ($/Mg recycled)
c_rec = 40
# Transportation cost ($/Mg·km)
t = 1.5

# Distances (km) from cities to facilities
d = Dict(
    (1,:LF)=>5, (1,:MRF)=>30, (1,:WTE)=>15,
    (2,:LF)=>15, (2,:MRF)=>25, (2,:WTE)=>10,
    (3,:LF)=>13, (3,:MRF)=>45, (3,:WTE)=>20)
# Inter-facility distances
d_inter = Dict(
    (:MRF,:LF)=>32, (:MRF,:WTE)=>15, (:WTE,:LF)=>18)
# MRF recycling fraction
R_MRF = 0.3775 
# WTE ash fractions
a_raw = 0.1641
a_res = 0.14


# Build model
model = Model(HiGHS.Optimizer)
# Decision variables
@variable(model, y[f in facilities], Bin)
@variable(model, x[i in cities, f in facilities] >= 0)
@variable(model, r_MRF_LF >= 0)
@variable(model, r_MRF_WTE >= 0)
@variable(model, a_WTE_LF >= 0)
# Auxiliary variables
@expression(model, m_MRF, sum(x[i,:MRF] for i in cities))
@expression(model, m_WTE, sum(x[i,:WTE] for i in cities) + r_MRF_WTE)
@expression(model, m_LF, sum(x[i,:LF] for i in cities) + r_MRF_LF + a_WTE_LF)
@expression(model, m_rec, R_MRF*m_MRF)
@expression(model, m_res, (1-R_MRF)*m_MRF)


# Constraints
# 1) City supply balances
for i in cities
    @constraint(model, sum(x[i,f] for f in facilities) == S[i])
end
# 2) MRF residual allocation
@constraint(model, r_MRF_LF + r_MRF_WTE == m_res)
# 3) WTE ash
@constraint(model, a_WTE_LF == a_raw*sum(x[i,:WTE] for i in cities) + a_res*r_MRF_WTE)
# 4) Capacity constraints
@constraint(model, m_LF <= C[:LF]*y[:LF])
@constraint(model, m_MRF <= C[:MRF]*y[:MRF])
@constraint(model, m_WTE <= C[:WTE]*y[:WTE])
# 5) Linking constraints (prevent flow to closed facilities)
for f in facilities
    @constraint(model, sum(x[i,f] for i in cities) <= C[f]*y[f])
end
@constraint(model, r_MRF_LF <= C[:LF]*y[:LF])
@constraint(model, r_MRF_WTE <= C[:WTE]*y[:WTE])
@constraint(model, a_WTE_LF <= C[:LF]*y[:LF])


# Objective function
@expression(model, obj,
    # Fixed costs
    sum(F[f]*y[f] for f in facilities) +
    # Tipping raw MSW
    sum(c[f]*x[i,f] for i in cities, f in facilities) +
    # Recycling processing
    c_rec * m_rec +
    # MRF residual tipping
    c[:LF]*r_MRF_LF + c[:WTE]*r_MRF_WTE +
    # WTE ash tipping
    c[:LF]*a_WTE_LF +
    # Transportation: cities to facilities
    t*sum(x[i,f]*d[(i,f)] for i in cities, f in facilities) +
    # Transportation: inter-facility / ash
    t*(r_MRF_LF*d_inter[:MRF,:LF] + r_MRF_WTE*d_inter[:MRF,:WTE] + a_WTE_LF*d_inter[:WTE,:LF])
)

@objective(model, Min, obj)

# Solve
optimize!(model)

# Results
status = termination_status(model)
println("Solver status: ", status)

if status == MOI.OPTIMAL
    println("Optimal total daily cost: \$", objective_value(model))
    println("Facility openings:")
    for f in facilities
        println("  ", f, ": ", value(y[f]))
    end
    println("City-to-facility flows:")
    for i in cities, f in facilities
        println("  City ", i, " → ", f, ": ", value(x[i,f]))
    end
    println("MRF residuals → LF: ", value(r_MRF_LF))
    println("MRF residuals → WTE: ", value(r_MRF_WTE))
    println("WTE ash → LF: ", value(a_WTE_LF))
end


Running HiGHS 1.12.0 (git hash: 755a8e027): Copyright (c) 2025 HiGHS under MIT licence terms
MIP has 14 rows; 15 cols; 52 nonzeros; 3 integer variables (3 binary)
Coefficient ranges:
  Matrix  [1e-01, 4e+02]
  Cost    [6e+01, 2e+03]
  Bound   [1e+00, 1e+00]
  RHS     [9e+01, 1e+02]
Presolving model
14 rows, 15 cols, 52 nonzeros  0s
10 rows, 13 cols, 40 nonzeros  0s
Presolve reductions: rows 10(-4); columns 13(-2); nonzeros 40(-12) 

Solving MIP model with:
   10 rows
   13 cols (2 binary, 0 integer, 0 implied int., 11 continuous, 0 domain fixed)
   40 nonzeros

Src: B => Branching; C => Central rounding; F => Feasibility pump; H => Heuristic;
     I => Shifting; J => Feasibility jump; L => Sub-MIP; P => Empty MIP; R => Randomized rounding;
     S => Solve LP; T => Evaluate node; U => Unbounded; X => User solution; Y => HiGHS solution;
     Z => ZI Round; l => Trivial lower; p => Trivial point; u => Trivial upper; z => Trivial zero

        Nodes      |    B&B Tree     |            Obje

This code uses the Decision variables, Auxiliary variables, Constraints, and Objective function from the previous question through the optimization model. 

#### Problem 1.6

Draw a diagram showing the flows of waste between the cities and the
facilities. Which facilities (if any) will not be used? Does this
solution make sense?

City1 (100)  ----->  LF (open)

City2 (90)   ----->  WTE (open)  --(ash 21.59)--> LF

City3 (41.59)-> WTE

City3 (79.04) -> LF

MRF : CLOSED (no flows)

### Problem 2 (6 points)

Consider a two-period economic dispatch problem, based on the
multi-period example from Lecture 14 (on 10/29). The generator data,
including ramping constraints for each generator, is provided in
\`data/generators.csv.’ In period 1, the demand is
$d_1 = 1100 \text{MW}$. In period 2, the demand is projected to be
$d_2 = 1200 \text{MW}$, but there is a 25% probability that it is \$1500
. In the first period, the solar capacity factor is $0.9$ and the wind
capacity factor is $0.45$, but in the second period, there is some
uncertainty: the forecasted solar and wind capacity factors are $0.95$
and $0.4$, respectively, but there is a 30% probability that they are
$0.75$ and $0.5$. Your goal is to identify how to dispatch your
generators to minimize the cost of meeting demand.

#### Problem 2.1

Draw a scenario tree for this problem.

                                  ROOT (time 0)
                                     |
                                     |  (deterministic period 1)
                                     v
                         Node t=1: d1 = 1100 MW, solar CF=0.90, wind CF=0.45
                                     |
                ---------------------------------------------------------
                |                         |                       |     
                |                         |                       |
        (demand branch)            (demand branch)           (conceptually same
        d2=1200 (0.75)             d2=1500 (0.25)           single node at t=1)
                |                         |
         -------------------         -------------------------
         |         |       |            |       |         |
        (renew) (renew) (renew)     (renew)  (renew)    (renew)
         0.70    0.30             0.70        0.30
         |        |                |           |
         |        |                |           |
      Scen A    Scen B           Scen C      Scen D
 
       (1200,    (1200,           (1500,     (1500,
 
       0.95/0.40) 0.75/0.50)       0.95/0.40) 0.75/0.50)
 
       prob=0.525  prob=0.225      prob=0.175  prob=0.075


#### Problem 2.2

Formulate a stochastic linear program for this problem based on your
scenario tree from Problem 2.1 and the data in `data/generators.csv`.

In [ ]:
gen = CSV.read("data/generators.csv", DataFrame)

# Define sets and parameters
G = Vector{String}(gen.Plant)           
Pmin = Dict(gen.Plant .=> gen.Pmin)
Pmax = Dict(gen.Plant .=> gen.Pmax)
VarCost = Dict(gen.Plant .=> gen.VarCost)
Ramp = Dict(gen.Plant .=> gen.Ramp)

# Identify solar and wind units
solar_units = gen.Plant[gen.Resource .== "solar"]
wind_units  = gen.Plant[gen.Resource .== "wind"]

# SCENARIOS AND DATA
T = 1:2                                  # time periods
S = 1:2                                  # 2 end scenarios for period 2

# Scenario probabilities
pi = Dict(1 => 0.75,  # normal demand, normal renewables
         2 => 0.25)  # high demand, stressed renewables

# Demands
d = Dict((1,1)=>1100.0, (1,2)=>1100.0,  # period 1 deterministic
         (2,1)=>1200.0, (2,2)=>1500.0)  # period 2 uncertain

# Solar and wind capacity factors
solar_cf = Dict((1,1)=>0.90, (1,2)=>0.90,
                (2,1)=>0.95, (2,2)=>0.75)

wind_cf  = Dict((1,1)=>0.45, (1,2)=>0.45,
                (2,1)=>0.40, (2,2)=>0.50)


model = Model(HiGHS.Optimizer)

# DECISION VARIABLES
# Generation per generator, period, and scenario
p = @variable(model, [g in G, t in T, s in S])

# CONSTRAINTS
#Power limits: Pmin <= p <= Pmax
@constraint(model, [g in G, t in T, s in S],
    Pmin[g] <= p[g,t,s] <= Pmax[g])

#Renewable availability (capacity factor)
@constraint(model, [g in solar_units, t in T, s in S], p[g,t,s] <= Pmax[g] * solar_cf[(t,s)])
@constraint(model, [g in wind_units, t in T, s in S], p[g,t,s] <= Pmax[g] * wind_cf[(t,s)])

#Ramping constraints period 1 to 2
@constraint(model, [g in G, s in S], p[g,2,s] - p[g,1,s] <= Ramp[g])
@constraint(model, [g in G, s in S], p[g,1,s] - p[g,2,s] <= Ramp[g])

#Supply-demand balance for each period, each scenario
@constraint(model, [t in T, s in S], sum(p[g,t,s] for g in G) == d[(t,s)])

#Minimize expected generation cost
@objective(model, Min, sum(pi[s] * sum(VarCost[g] * p[g,t,s] for g in G) for t in T, s in S))

println("Stochastic economic dispatch model formulated successfully.")


┌ Warning: thread = 1 warning: parsed expected 6 columns, but didn't reach end of line around data row: 3. Parsing extra columns and widening final columnset
└ @ CSV C:\Users\shini\.julia\packages\CSV\XLcqT\src\file.jl:593
┌ Warning: thread = 1 warning: only found 6 / 7 columns around data row: 4. Filling remaining columns with `missing`
└ @ CSV C:\Users\shini\.julia\packages\CSV\XLcqT\src\file.jl:592
┌ Warning: thread = 1 warning: only found 6 / 7 columns around data row: 5. Filling remaining columns with `missing`
└ @ CSV C:\Users\shini\.julia\packages\CSV\XLcqT\src\file.jl:592
┌ Warning: thread = 1 warning: only found 6 / 7 columns around data row: 6. Filling remaining columns with `missing`
└ @ CSV C:\Users\shini\.julia\packages\CSV\XLcqT\src\file.jl:592


Stochastic economic dispatch model formulated successfully.


## References

List any external references consulted, including classmates.